In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

#PROJECT_ROOT = Path(
#    "/media/ehsan/MyFiles3/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting"
#)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import load_config

cfg = load_config(PROJECT_ROOT / "configs/config.yaml")

cfg["data"]["root"] = (
    "/media/ehsan/MyFiles3/FraUAS/HIS Project/PTB-XL"
)

print(cfg["data"])

from data import prepare

cache, manifest = prepare(cfg)

print("Cache:", cache)
print("Splits:", manifest["splits"])

{'root': '/media/ehsan/MyFiles3/FraUAS/HIS Project/PTB-XL', 'sampling_rate': 100, 'history': 700, 'horizon': 300, 'synthetic': False, 'start': 0, 'layout': 'time_first'}
Cache: data/study_v3/ee7d4f585e459326
Splits: {'train': {'records': 339, 'patients': 300, 'windows': 339}, 'val': {'records': 81, 'patients': 75, 'windows': 81}}


In [3]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/media/ehsan/MyFiles3/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting"
)

print(PROJECT_ROOT.exists())
print((PROJECT_ROOT / "src").exists())
print(PROJECT_ROOT / "src")

import sys
print(sys.executable)

True
True
/media/ehsan/MyFiles3/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting/src
/home/ehsan/.venvs/ptbxl_tsa_venv/bin/python3


In [10]:
from data import ECGDataset
from preprocessing import feature_arrays

arrays, names, details = feature_arrays(
    cfg,
    cache,
    "F0",
    splits=("train", "val"),
)

dataset = ECGDataset(
    cache,
    cfg,
    "train",
    "P0",
    arrays["train"],
)

sample = dataset[0]

print("x:", sample["x"].shape)
print("y:", sample["y"].shape)
print("static:", sample["static"].shape)

x: torch.Size([700, 12])
y: torch.Size([300, 12])
static: torch.Size([0])


In [11]:
sample = dataset[0]

print("x:", sample["x"].shape)
print("y:", sample["y"].shape)

x: torch.Size([700, 12])
y: torch.Size([300, 12])


In [12]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False
)

batch = next(iter(loader))

print("x batch:", batch["x"].shape)
print("y batch:", batch["y"].shape)

x batch: torch.Size([4, 700, 12])
y batch: torch.Size([4, 300, 12])


In [4]:
import torch

x = torch.randn(2, 700, 12)

patches = x.unfold(
    dimension=1,
    size=16,
    step=8
)

print(patches.shape)

torch.Size([2, 86, 12, 16])


In [5]:
from models import build_model
import torch

model = build_model(
    "tft",
    static_dim=0,
    horizon=cfg["data"]["horizon"],
    **cfg["model"]
)

x = torch.randn(2,700,12)

with torch.no_grad():
    projected = model.input_projection(x)

print(projected.shape)

torch.Size([2, 700, 64])


In [6]:
from models.PatchTST_32 import PatchTST32

print(PatchTST32)

<class 'models.PatchTST_32.PatchTST32'>


In [8]:
from train import train_run

run = train_run(
    cfg,
    cache,
    architecture="patchtst_32",
    preprocessing="P0",
    feature_set="F0",
    epochs=1,
    name="patchtst32_smoke"
)

patchtst32_smoke: epoch 1/1; validation patient MAE 0.115598 mV
